In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import pickle

In [ ]:
df = pd.read_csv("House_Rent_Dataset.csv")
df.head()

,Posted On,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact
0,2022-05-18,2,10000,1100,Ground out of 2,Super Area,Bandel,Kolkata,Unfurnished,Bachelors/Family,2,Contact Owner
1,2022-05-13,2,20000,800,1 out of 3,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
2,2022-05-16,2,17000,1000,1 out of 3,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
3,2022-07-04,2,10000,800,1 out of 2,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,1,Contact Owner
4,2022-05-09,2,7500,850,1 out of 2,Carpet Area,South Dum Dum,Kolkata,Unfurnished,Bachelors,1,Contact Owner


In [ ]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
def split_floor(x):
    x = str(x).lower()
    current_floor = None
    total_floor = None

    if "upper basement" in x:
        current_floor = -1
    elif "lower basement" in x:
        current_floor = -2
    elif "ground" in x:
        current_floor = 0

    # Now, try to extract total_floor if "out of" is present
    if "out of" in x:
        parts = x.split("out of")
        if len(parts) == 2:
            try:
                # If current_floor is still None, try to parse from the first part
                if current_floor is None:
                    current_floor_str = parts[0].strip()
                    if current_floor_str.isdigit() or (current_floor_str.startswith('-') and current_floor_str[1:].isdigit()):
                        current_floor = int(current_floor_str)

                total_floor = int(parts[1].strip())
            except ValueError:
                # Handle cases where conversion to int fails (e.g., '1 out of abc')
                pass

    return current_floor, total_floor

df["current_Floor"], df["Total_Floor"] = zip(*df["Floor"].apply(split_floor))
df.drop("Floor", axis=1, inplace=True)

In [ ]:
df["Rent_log"] = np.log(df["Rent"])

In [ ]:
df = pd.get_dummies(df, drop_first=True)
df.head()

,BHK,Rent,Size,Bathroom,current_Floor,Total_Floor,Rent_log,Posted On_2022-04-23,Posted On_2022-04-24,Posted On_2022-04-25,...,City_Delhi,City_Hyderabad,City_Kolkata,City_Mumbai,Furnishing Status_Semi-Furnished,Furnishing Status_Unfurnished,Tenant Preferred_Bachelors/Family,Tenant Preferred_Family,Point of Contact_Contact Builder,Point of Contact_Contact Owner
0,2,10000,1100,2,0.0,2.0,9.210340,False,False,False,...,False,False,True,False,False,True,True,False,False,True
1,2,20000,800,1,1.0,3.0,9.903488,False,False,False,...,False,False,True,False,True,False,True,False,False,True
2,2,17000,1000,1,1.0,3.0,9.740969,False,False,False,...,False,False,True,False,True,False,True,False,False,True
3,2,10000,800,1,1.0,2.0,9.210340,False,False,False,...,False,False,True,False,False,True,True,False,False,True
4,2,7500,850,1,1.0,2.0,8.922658,False,False,False,...,False,False,True,False,False,True,False,False,False,True


In [ ]:
X = df.drop(["Rent_log", "Rent"], axis=1)
y = df["Rent_log"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
num_cols = ['current_Floor', 'Total_Floor']

imputer = SimpleImputer(strategy="median")
X_train[num_cols] = imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = imputer.transform(X_test[num_cols])

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_leaf': [1, 3, 5]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train_scaled, y_train)
best_rf = grid.best_estimator_

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Best Parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'n_estimators': 200}
Best CV Score: 0.8203470173844705


In [ ]:
def evaluate(model, name, X_test_used):
    pred = model.predict(X_test_used)
    r2 = r2_score(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))

    print(f"\n{name} Performance:")
    print("R2:", r2)
    print("MAE:", mae)
    print("RMSE:", rmse)

evaluate(lr, "Linear Regression", X_test_scaled)
evaluate(rf, "Random Forest Base", X_test_scaled)
evaluate(best_rf, "Random Forest Tuned", X_test_scaled)


Linear Regression Performance:
R2: 0.23576429642552776
MAE: 0.5157625239656998
RMSE: 0.8118775358677026

Random Forest Base Performance:
R2: 0.8493747130475306
MAE: 0.2692814250918331
RMSE: 0.3604340459860366

Random Forest Tuned Performance:
R2: 0.8488723897913062
MAE: 0.2709632655697883
RMSE: 0.3610345550638758


In [ ]:
print("\nRandom Forest Tuned")
print("Train R2:", best_rf.score(X_train_scaled, y_train))
print("Test R2:", best_rf.score(X_test_scaled, y_test))


Random Forest Tuned
Train R2: 0.9639712504212995
Test R2: 0.8488723897913062


In [ ]:
pickle.dump(best_rf, open("rent_model.pkl", "wb"))
pickle.dump(imputer, open("imputer.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))
pickle.dump(X.columns.tolist(), open("columns.pkl", "wb"))

In [ ]:
model = pickle.load(open("rent_model.pkl", "rb"))
imputer = pickle.load(open("imputer.pkl", "rb"))
scaler = pickle.load(open("scaler.pkl", "rb"))
cols = pickle.load(open("columns.pkl", "rb"))

In [ ]:
def predict_rent(data_dict):
    # Create a DataFrame with all expected columns from 'cols', initialized to 0.
    # This ensures all one-hot encoded features not in data_dict are set to 0
    # and all features the model was trained on are present.
    input_df = pd.DataFrame(0, index=[0], columns=cols)

    # Populate the DataFrame with the provided data_dict values
    for key, value in data_dict.items():
        if key in input_df.columns:
            input_df.loc[0, key] = value
        # else: Ignore keys in data_dict that are not in cols to prevent errors from extra user input

    x = input_df # Now x has all the required columns in the correct order

    # Impute floors (only for 'current_Floor', 'Total_Floor')
    x[['current_Floor','Total_Floor']] = imputer.transform(
        x[['current_Floor','Total_Floor']]
    )

    # Scale numerical features
    # Note: If 'Rent' is still in 'cols' (due to data leakage), its value will be 0 after initialization.
    # The scaler was fitted on X which included 'Rent'. This is part of the data leakage problem.
    # The ideal fix involves dropping 'Rent' from X before training, which would also remove it from 'cols'.
    x_scaled = scaler.transform(x)

    # Predict log-rent
    pred_log = model.predict(x_scaled)[0]

    # Convert back to actual rent
    return np.exp(pred_log)

In [ ]:
sample_input = {
    'BHK': 2,
    'Size': 1100,
    'Bathroom': 2,
    'current_Floor': 1,
    'Total_Floor': 5,
    'Area Type_Super Area': 1,
    'Area Type_Carpet Area': 0,
    'Area Type_Built Area': 0,
    'City_Mumbai': 1,
    'City_Bangalore': 0,
    'City_Delhi': 0,
    'City_Hyderabad': 0,
    'City_Kolkata': 0,
    'furnishing Status_Unfurnished': 0,
    'furnishing Status_Semi-Furnished': 1,
    'Tenant Preferred_Bachelors': 0,
    'Tenant Preferred_Family': 1,
    'Point of Contact_Owner': 1,
    'Point of Contact_Agent': 0
}

predicted_rent = predict_rent(sample_input)
print("Predicted Rent (₹):", predicted_rent)

Predicted Rent (₹): 130099.75440985414
